# ECG Project — Phase 2: PDF Parameter Extraction

This notebook extracts printed ECG measurements and machine findings from an ECG PDF.

It first uses embedded PDF text and falls back to OCR only when necessary.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
!pip install -q pypdf pdf2image pytesseract pandas pillow


## 1. Upload an ECG PDF

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("Please upload an ECG PDF.")

pdf_path = Path(pdf_files[0])
print("Selected:", pdf_path)


## 2. Extract embedded text, with OCR fallback

In [ ]:
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract

def extract_embedded_text(path):
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages).strip()

def extract_ocr_text(path, dpi=300):
    pages = convert_from_path(str(path), dpi=dpi)
    parts = []
    for i, image in enumerate(pages, start=1):
        image.save(f"ecg_page_{i}.png")
        parts.append(pytesseract.image_to_string(image, config="--psm 6"))
    return "\n".join(parts).strip()

embedded_text = extract_embedded_text(pdf_path)

if len(embedded_text) >= 80:
    raw_text = embedded_text
    extraction_method = "embedded_pdf_text"
else:
    raw_text = extract_ocr_text(pdf_path)
    extraction_method = "ocr_fallback"

print("Method:", extraction_method)
print(raw_text[:3000])


## 3. Parse the ECG report

In [ ]:
import re
import json

def normalize_text(text):
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r"[\t\r]+", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def first_match(pattern, text, cast=float, flags=re.IGNORECASE):
    match = re.search(pattern, text, flags)
    if not match:
        return None
    try:
        return cast(match.group(1))
    except (ValueError, TypeError):
        return None

def parse_axes(text):
    m = re.search(
        r"P\s*/\s*QRS\s*/\s*T\s*[:\-]?\s*([+-]?\d+(?:\.\d+)?)\s*/\s*([+-]?\d+(?:\.\d+)?)\s*/\s*([+-]?\d+(?:\.\d+)?)",
        text, re.IGNORECASE
    )
    return (None, None, None) if not m else tuple(float(m.group(i)) for i in range(1, 4))

def parse_qt_qtc(text):
    m = re.search(
        r"QT\s*/\s*QTc(?:Bz)?\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)\s*ms",
        text, re.IGNORECASE
    )
    return (None, None) if not m else (float(m.group(1)), float(m.group(2)))

def parse_rv5_sv1(text):
    m = re.search(
        r"RV5\s*/\s*SV1\s*[:\-]?\s*([+-]?\d+(?:\.\d+)?)\s*/\s*([+-]?\d+(?:\.\d+)?)\s*mV",
        text, re.IGNORECASE
    )
    return (None, None) if not m else (float(m.group(1)), float(m.group(2)))

def extract_diagnosis_lines(text):
    m = re.search(
        r"Diagnosis\s+Information\s*:\s*(.*?)(?:Report\s+Confirmed\s+by|\d{2}-\d{2}-\d{4}|$)",
        text, re.IGNORECASE | re.DOTALL
    )
    if not m:
        return []
    lines = []
    for line in m.group(1).splitlines():
        line = re.sub(r"\s+", " ", line).strip(" :-")
        if len(line) > 2:
            lines.append(line)
    return lines

def parse_ecg_report(text, method):
    text = normalize_text(text)
    qt_ms, qtc_ms = parse_qt_qtc(text)
    p_axis, qrs_axis, t_axis = parse_axes(text)
    rv5, sv1 = parse_rv5_sv1(text)
    diagnosis_lines = extract_diagnosis_lines(text)

    rhythm = None
    findings = []
    for item in diagnosis_lines:
        if rhythm is None and re.search(r"\brhythm\b", item, re.IGNORECASE):
            rhythm = item
        else:
            findings.append(item)

    return {
        "heart_rate_bpm": first_match(r"\bHR\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*bpm", text),
        "p_duration_ms": first_match(r"(?m)^\s*P\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text),
        "pr_interval_ms": first_match(r"\bPR\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text),
        "qrs_duration_ms": first_match(r"\bQRS\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*ms", text),
        "qt_interval_ms": qt_ms,
        "qtc_bazett_ms": qtc_ms,
        "p_axis_deg": p_axis,
        "qrs_axis_deg": qrs_axis,
        "t_axis_deg": t_axis,
        "rv5_mv": rv5,
        "sv1_mv": sv1,
        "rhythm": rhythm,
        "machine_findings": findings,
        "paper_speed_mm_s": first_match(r"(\d+(?:\.\d+)?)\s*mm/s", text),
        "gain_mm_mv": first_match(r"(\d+(?:\.\d+)?)\s*mm/mV", text),
        "report_datetime": first_match(
            r"(\d{2}-\d{2}-\d{4}\s+\d{1,2}:\d{2}:\d{2}\s*(?:AM|PM)?)",
            text, cast=str
        ),
        "extraction_method": method
    }

ecg_data = parse_ecg_report(raw_text, extraction_method)
print(json.dumps(ecg_data, indent=2))


## 4. Validate extracted values

In [ ]:
def validate_ecg_data(data):
    warnings = []
    ranges = {
        "heart_rate_bpm": (20, 250),
        "p_duration_ms": (40, 200),
        "pr_interval_ms": (60, 400),
        "qrs_duration_ms": (40, 250),
        "qt_interval_ms": (150, 700),
        "qtc_bazett_ms": (150, 700),
        "p_axis_deg": (-180, 180),
        "qrs_axis_deg": (-180, 180),
        "t_axis_deg": (-180, 180),
        "rv5_mv": (0, 10),
        "sv1_mv": (0, 10),
        "paper_speed_mm_s": (10, 100),
        "gain_mm_mv": (2, 40),
    }

    for field, (low, high) in ranges.items():
        value = data.get(field)
        if value is None:
            warnings.append(f"Missing field: {field}")
        elif not low <= value <= high:
            warnings.append(f"Out-of-range value: {field}={value}")

    if not data.get("rhythm"):
        warnings.append("Rhythm was not extracted.")

    return warnings

validation_warnings = validate_ecg_data(ecg_data)

if validation_warnings:
    print("\n".join("- " + item for item in validation_warnings))
else:
    print("No validation warnings.")


## 5. Save JSON and CSV

In [ ]:
import pandas as pd

final_output = {
    "source_file": pdf_path.name,
    "ecg_parameters": ecg_data,
    "validation_warnings": validation_warnings
}

json_path = Path("/content/ecg_extracted_parameters.json")
csv_path = Path("/content/ecg_extracted_parameters.csv")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

csv_row = ecg_data.copy()
csv_row["machine_findings"] = " | ".join(csv_row.get("machine_findings", []))
csv_row["source_file"] = pdf_path.name
csv_row["validation_warnings"] = " | ".join(validation_warnings)

pd.DataFrame([csv_row]).to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)


## 6. Download outputs

In [ ]:
from google.colab import files

files.download("/content/ecg_extracted_parameters.json")
files.download("/content/ecg_extracted_parameters.csv")


## Expected result for your sample format

The sample report should approximately return:

```json
{
  "heart_rate_bpm": 87,
  "p_duration_ms": 112,
  "pr_interval_ms": 160,
  "qrs_duration_ms": 87,
  "qt_interval_ms": 372,
  "qtc_bazett_ms": 448,
  "p_axis_deg": 69,
  "qrs_axis_deg": 40,
  "t_axis_deg": 78,
  "rv5_mv": 2.572,
  "sv1_mv": 0.756,
  "rhythm": "Sinus Rhythm"
}
```

This extracts the machine-reported values; it does not independently verify the diagnosis.
